# Train logo/field detector (YOLOv8) on Colab

Mirrors `train_logo.py` from the `YOLO_Model` branch, but running on Colab's free GPU instead of CPU.

**Before running:** Runtime menu -> Change runtime type -> Hardware accelerator -> GPU -> Save.

In [ ]:
# 1. Confirm you actually got a GPU
!nvidia-smi

In [ ]:
# 2. Install dependencies
!pip install -q ultralytics

In [ ]:
# 3. Upload final_dataset.zip (zip your local final_dataset/ folder first, see chat instructions)
from google.colab import files
uploaded = files.upload()  # a file-picker dialog will appear here

In [ ]:
# 4. Unzip it into /content/final_dataset
import zipfile, os

zip_name = next(iter(uploaded))  # picks up whatever filename you uploaded
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

print(os.listdir('/content/final_dataset'))

In [ ]:
# 5. Fix the hardcoded Windows path inside logo_dataset.yaml
yaml_path = "/content/final_dataset/logo_dataset.yaml"
content = open(yaml_path).read()
content = content.replace("C:/Users/xian.xin/MarkLogic/final_dataset", "/content/final_dataset")
open(yaml_path, "w").write(content)
print(open(yaml_path).read())

In [ ]:
# 6. Train - stronger settings since we have GPU headroom (yolov8m + bigger batch)
from ultralytics import YOLO
import torch

device = 0 if torch.cuda.is_available() else "cpu"
print("Training on:", "GPU" if device == 0 else "CPU")

model = YOLO("yolov8m.pt")  # bumped up from yolov8s.pt now that compute is cheap

model.train(
    name="train-colab",
    exist_ok=True,
    data=yaml_path,
    epochs=150,
    imgsz=640,
    batch=16,           # bumped up from 8 - Colab's T4 has far more VRAM than 4GB
    device=device,
    workers=4,
    amp=True,
    cache=True,
    patience=30,
    val=True,
    warmup_epochs=5,
    optimizer="auto",
    augment=True,
    flipud=0.1,
    fliplr=0.5,
    mosaic=1.0,
    degrees=5.0,
    scale=0.3,
)

In [ ]:
# 7. Download the trained weights back to your PC
from google.colab import files
files.download("runs/detect/train-colab/weights/best.pt")